基本的なものをimport,または定義していく

* **概要**: 1990年代のインターネット掲示板（ニュースグループ）の投稿を集めたデータセットです。
* **データ数**: 約18,000件のテキスト文書。
異なるカテゴリーの記事を4つ,そして関係ないものを取り除く
datesetの中身(Bunch)は、dataset.data:テキストの集合,dataset.target:カテゴリの数値ラベル,dataset.target_names:カテゴリの名前リスト見たいな感じ

In [94]:
import numpy as np
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.metrics import adjusted_rand_score
import numpy as np
np.random.seed(25)  

In [95]:
categories = ['alt.atheism', 'comp.graphics', 'sci.space', 'rec.sport.baseball']

# データの取得（ヘッダーやフッターなどのノイズを除去して純粋なテキストのみを取得）
print("Loading dataset...")
dataset = fetch_20newsgroups(subset='train', categories=categories,
                             shuffle=True, random_state=42,
                             remove=('headers', 'footers', 'quotes'))
raw_texts = dataset.data
print(f"Loaded {len(raw_texts)} documents.")

Loading dataset...
Loaded 2254 documents.


ここは自作しません.
countvectorizerで、
95%以上の文章に存在する単語は無視する、２個未満の単語は無視する。頻出頻度の高い単語(1000)に絞る,意味のない単語は消す
Xは、疎行列.もし普通に行列を作ると(d*V)の数になるので、メモリが破綻する。そのため、0の記録はしない。1の記録のみする(カウント数がある)
vocabは辞書こと。numpyの配列で渡されている

In [96]:
vectorizer = CountVectorizer(max_df=0.95, min_df=2,
                             max_features=1000,
                             stop_words='english')


X = vectorizer.fit_transform(raw_texts)

# 辞書（IDから実際の単語文字列へのマッピング）を取得
vocab = vectorizer.get_feature_names_out()

In [97]:
X[1:15].shape

(14, 1000)

実装してみた

In [98]:
import numpy as np
from scipy.special import digamma
import scipy.sparse as sp

class CustomLDA_SVI:
    def __init__(self, n_topics=4, alpha=0.1, eta=0.1,
                 tau=1024.0, kappa=0.7, batch_size=128, max_iter=10):

        self.K = n_topics
        self.alpha = alpha
        self.eta = eta
        self.tau = tau#学習率をある程度調整しておく
        self.kappa = kappa
        self.batch_size = batch_size
        self.max_iter = max_iter
        self.lambda_param = None
        self.t = 0 # 現在のイテレーション回数 (学習率の減衰に使用)

    def _init_params(self, vocab_size):
        self.V = vocab_size
        self.lambda_param = np.random.gamma(100., 1./100., (self.K, self.V))#4*1000の行列を作成,初期カウントは1くらいに設定

    def fit(self, X):
        D, V = X.shape
        if self.lambda_param is None:
            self._init_params(V)

        for epoch in range(self.max_iter):
            for batch_idx in range(0, D, self.batch_size):
                X_batch = X[batch_idx : batch_idx + self.batch_size]

                gamma_batch, lambda_hat = self._cavi_local_update(X_batch, D)
                rho_t = (self.t + self.tau) ** (-self.kappa) # 2. 学習率(ステップサイズ) ρ_t の計算
                self.lambda_param = (1 - rho_t) * self.lambda_param + rho_t * lambda_hat # ローバル更新: λ の更新

                self.t += 1

    def _cavi_local_update(self, X_batch, total_D):
        B = X_batch.shape[0]
        gamma = np.ones((B, self.K)) * self.alpha #size設定、(ミニバッチ、トピック数),ここは工夫Point,ある程度ganmaを大きくする。ただし、どのくらいがいいのかは記事による。今回はなし。
        #gamma = np.random.gamma(100., 1./100., (B, self.K))

        lambda_hat = np.zeros((self.K, self.V))
        E_log_beta = digamma(self.lambda_param) - digamma(self.lambda_param.sum(axis=1))[:, np.newaxis]# Ψ(λ_{kv}) - Ψ(Σ λ_{kv}) の事前計算,K*Vの行列,sumのところは4だけど、次元追加
        X_csr = X_batch.tocsr()#スライスすると、型が分かることがあるらしいので

        for d in range(B):#ミニバッチ(文章)ごとに解析
            row = X_csr[d]
            word_indices = row.indices#位置を取得
            counts = row.data #回数を取得

            if len(word_indices) == 0:
                continue

            gamma_d = gamma[d, :].copy()#k行のベクトル

            for local_iter in range(20):
                #ここに部分は論文で比例で書いているが，それを正規化できるようにする.あとはφはかませだから，なるべく軽くなるようにする
                E_log_theta_d = digamma(gamma_d) - digamma(gamma_d.sum())
                log_phi_dn = E_log_theta_d[np.newaxis, :] + E_log_beta[:, word_indices].T   #[1,4]と[4,1000]   単語の種類*トッピッく数

                phi_dn = np.exp(log_phi_dn - log_phi_dn.max(axis=1, keepdims=True)) #logsumで，Inf対策
                phi_dn /= phi_dn.sum(axis=1, keepdims=True)#正規化

                gamma_d_new = self.alpha + (counts[:, np.newaxis] * phi_dn).sum(axis=0)#更新kベクトル，単語方向に和を出す

                if np.abs(gamma_d_new - gamma_d).mean() < 1e-3:#収束判定,短い文のため
                    gamma_d = gamma_d_new
                    break
                gamma_d = gamma_d_new

            gamma[d, :] = gamma_d

            for i, word_idx in enumerate(word_indices):
                lambda_hat[:, word_idx] += counts[i] * phi_dn[i, :]

        lambda_hat = self.eta + (total_D / B) * lambda_hat #sviの部分

        return gamma, lambda_hat

    #def transform(self, X):
        # グローバルパラメータ λ は固定したまま、_cavi_local_update と同じ計算を行い、
        # 正規化した γ を返す（これが θ に相当）
        #pass

    def get_top_words(self, vocab, n_top_words=5):
        if self.lambda_param is None:
            print("まだ学習されていません！先に fit() を実行してください。")
            return

        print(f"【各トピックの上位 {n_top_words} 単語】")
        for k in range(self.K):
            top_word_indices = np.argsort(self.lambda_param[k, :])[::-1][:n_top_words]
            top_words = [vocab[idx] for idx in top_word_indices]
            
            # 3. 結果をプリント
            print(f"Topic {k}: {', '.join(top_words)}")
    
    def transform(self, X):
        if self.lambda_param is None:
            raise ValueError("まだモデルが学習されていません！先に fit() を実行してください。")

        D = X.shape[0]
        gamma_result, _ = self._cavi_local_update(X, total_D=D)
        
        # 2. 数式上のカウント数（gamma）を、合計 1.0 の「確率分布（theta）」に変換する
        #    行方向（axis=1）に合計を出して、ブロードキャストで割り算
        theta = gamma_result / gamma_result.sum(axis=1, keepdims=True)
        
        return theta

その結果、pythonのライブライと近いものができた.乱数は1~50で変化させてみる.正答率は,ARIで判断(ランダムに記事を２個選んで、その２つが正しい別れ方をしていたら1点のようにする,まぐれで当たる分もあるのでそこはマイナスにしてある)


In [99]:
result=[]
for seed in range(50):
    np.random.seed(seed)
    lda = CustomLDA_SVI()
    lda.fit(X)
    theta = lda.transform(X)
    predicted_topics = np.argmax(theta, axis=1)
    score = adjusted_rand_score(dataset.target, predicted_topics) 
    result.append(score)
print(np.mean(result))

0.21927994401983844


※補足:各トピックの上位 5 単語の見方。

In [100]:
lda.get_top_words(vocab,5)

【各トピックの上位 5 単語】
Topic 0: don, think, just, people, like
Topic 1: 10, 000, 333, 500, cubs
Topic 2: jpeg, file, gif, image, format
Topic 3: space, edu, data, image, graphics
